# KPIs para Dashboard

## Célula 1 — Configuração

In [1]:
from pyspark.sql import functions as F

GOLD_PATH = "s3a://gold/tse"

print(f"Gold Path: {GOLD_PATH}")

Gold Path: s3a://gold/tse


## Célula 2 — Carregar tabelas Gold

In [2]:
fato_candidatura_dashboard = spark.read.format("delta").load(
    f"{GOLD_PATH}/fato_candidatura_dashboard"
)

fato_bem_candidato_dashboard = spark.read.format("delta").load(
    f"{GOLD_PATH}/fato_bem_candidato_dashboard"
)

dim_cargo = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_cargo"
)

dim_partido = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_partido"
)

dim_municipio = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_municipio"
)

26/06/23 16:36:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

## KPI 1 — Total de Candidatos

In [3]:
from pyspark.sql import functions as F

kpi_total_candidatos = (
    fato_candidatura_dashboard
    .agg(
        F.countDistinct("sq_candidato")
        .alias("total_candidatos")
    )
    .withColumn("data_referencia", F.current_timestamp())
)

kpi_total_candidatos.show()

26/06/23 16:36:51 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/06/23 16:36:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 13:=======================>                                  (2 + 3) / 5]

+----------------+--------------------+
|total_candidatos|     data_referencia|
+----------------+--------------------+
|           78349|2026-06-23 16:36:...|
+----------------+--------------------+



In [4]:
kpi_total_candidatos.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/kpi_total_candidatos")

## KPI 2 — Total de Partidos

In [5]:
kpi_total_partidos = (
    dim_partido
    .agg(
        F.countDistinct("numero_partido")
        .alias("total_partidos")
    )
    .withColumn("data_referencia", F.current_timestamp())
)

kpi_total_partidos.show()

+--------------+--------------------+
|total_partidos|     data_referencia|
+--------------+--------------------+
|            29|2026-06-23 16:37:...|
+--------------+--------------------+



In [6]:
kpi_total_partidos.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/kpi_total_partidos")

## KPI 3 — Total de Municípios

In [7]:
kpi_total_municipios = (
    dim_municipio
    .agg(
        F.count("*")
        .alias("total_municipios")
    )
    .withColumn("data_referencia", F.current_timestamp())
)

kpi_total_municipios.show()

+----------------+--------------------+
|total_municipios|     data_referencia|
+----------------+--------------------+
|             645|2026-06-23 16:37:...|
+----------------+--------------------+



In [8]:
kpi_total_municipios.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/kpi_total_municipios")

## KPI 4 — Bens Declarados

In [9]:
total_bens = (
    fato_bem_candidato_dashboard
    .agg(
        F.sum("valor_bem_candidato")
        .alias("valor_total_bens")
    )
)

In [10]:
media_bens = (
    fato_bem_candidato_dashboard
    .groupBy("sq_candidato")
    .agg(
        F.sum("valor_bem_candidato")
        .alias("total_bens_candidato")
    )
    .agg(
        F.avg("total_bens_candidato")
        .alias("media_bens_por_candidato")
    )
)

## Partido com maior patrimônio

In [11]:
patrimonio_partido = (
    fato_bem_candidato_dashboard
    .join(
        fato_candidatura_dashboard,
        "sq_candidato"
    )
    .groupBy("numero_partido")
    .agg(
        F.sum("valor_bem_candidato")
        .alias("valor_partido")
    )
)

In [12]:
partido_maior = (
    patrimonio_partido
    .join(
        dim_partido,
        "numero_partido"
    )
    .orderBy(F.desc("valor_partido"))
    .limit(1)
)

## Montagem final do KPI

In [13]:
kpi_bens_declarados = (
    total_bens
    .crossJoin(media_bens)
    .crossJoin(
        partido_maior.select(
            F.col("sigla_partido")
                .alias("partido_maior_patrimonio"),
            F.col("valor_partido")
                .alias("valor_partido_maior_patrimonio")
        )
    )
    .withColumn(
        "data_referencia",
        F.current_timestamp()
    )
)

kpi_bens_declarados.show(truncate=False)

+---------------------+------------------------+------------------------+------------------------------+--------------------------+
|valor_total_bens     |media_bens_por_candidato|partido_maior_patrimonio|valor_partido_maior_patrimonio|data_referencia           |
+---------------------+------------------------+------------------------+------------------------------+--------------------------+
|2.0410062838810017E10|414047.6090154999       |PRTB                    |3.2497325225799994E9          |2026-06-23 16:37:07.325156|
+---------------------+------------------------+------------------------+------------------------------+--------------------------+



In [14]:
kpi_bens_declarados.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/kpi_bens_declarados")

## Métrica 1 — Candidatos por Cargo

In [15]:
metrica_candidatos_por_cargo = (
    fato_candidatura_dashboard
    .groupBy("codigo_cargo")
    .agg(
        F.countDistinct("sq_candidato")
        .alias("total_candidatos")
    )
    .join(dim_cargo, "codigo_cargo")
    .select(
        "codigo_cargo",
        "descricao_cargo",
        "total_candidatos"
    )
    .orderBy(F.desc("total_candidatos"))
    .withColumn("data_referencia", F.current_timestamp())
)

metrica_candidatos_por_cargo.show(truncate=False)

+------------+---------------+----------------+-------------------------+
|codigo_cargo|descricao_cargo|total_candidatos|data_referencia          |
+------------+---------------+----------------+-------------------------+
|13          |VEREADOR       |74043           |2026-06-23 16:37:13.05642|
|12          |VICE-PREFEITO  |2169            |2026-06-23 16:37:13.05642|
|11          |PREFEITO       |2137            |2026-06-23 16:37:13.05642|
+------------+---------------+----------------+-------------------------+



In [16]:
metrica_candidatos_por_cargo.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/metrica_candidatos_por_cargo")

## Métrica 2 — Patrimônio por Partido

In [17]:
metrica_patrimonio_por_partido = (
    fato_bem_candidato_dashboard
    .join(fato_candidatura_dashboard, "sq_candidato")
    .groupBy("numero_partido")
    .agg(
        F.sum("valor_bem_candidato")
        .alias("patrimonio_declarado")
    )
    .join(dim_partido, "numero_partido")
    .select(
        "numero_partido",
        "sigla_partido",
        "patrimonio_declarado"
    )
    .orderBy(F.desc("patrimonio_declarado"))
    .limit(10)
    .withColumn("data_referencia", F.current_timestamp())
)

metrica_patrimonio_por_partido.show(truncate=False)

+--------------+-------------+--------------------+--------------------------+
|numero_partido|sigla_partido|patrimonio_declarado|data_referencia           |
+--------------+-------------+--------------------+--------------------------+
|28            |PRTB         |3.2497325225799994E9|2026-06-23 16:37:17.789229|
|22            |PL           |1.8374646678699996E9|2026-06-23 16:37:17.789229|
|55            |PSD          |1.7996912299499993E9|2026-06-23 16:37:17.789229|
|10            |REPUBLICANOS |1.76640245048E9     |2026-06-23 16:37:17.789229|
|20            |PODE         |1.7286705789999993E9|2026-06-23 16:37:17.789229|
|15            |MDB          |1.5626218354399996E9|2026-06-23 16:37:17.789229|
|44            |UNIÃO        |1.2491080968500004E9|2026-06-23 16:37:17.789229|
|11            |PP           |1.0996781811999998E9|2026-06-23 16:37:17.789229|
|40            |PSB          |8.2207584651E8      |2026-06-23 16:37:17.789229|
|45            |PSDB         |7.831199170500001E8 |2

In [18]:
metrica_patrimonio_por_partido.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/metrica_patrimonio_por_partido")

## Validação 

In [19]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/kpi_total_candidatos"
).show()

+----------------+--------------------+
|total_candidatos|     data_referencia|
+----------------+--------------------+
|           78349|2026-06-23 16:36:...|
+----------------+--------------------+



In [20]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/kpi_total_partidos"
).show()

+--------------+--------------------+
|total_partidos|     data_referencia|
+--------------+--------------------+
|            29|2026-06-23 16:37:...|
+--------------+--------------------+



In [21]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/kpi_total_municipios"
).show()

[Stage 202:>                                                        (0 + 1) / 1]

+----------------+--------------------+
|total_municipios|     data_referencia|
+----------------+--------------------+
|             645|2026-06-23 16:37:...|
+----------------+--------------------+



In [22]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/kpi_bens_declarados"
).show(truncate=False)

[Stage 212:>                                                        (0 + 1) / 1]

+---------------------+------------------------+------------------------+------------------------------+--------------------------+
|valor_total_bens     |media_bens_por_candidato|partido_maior_patrimonio|valor_partido_maior_patrimonio|data_referencia           |
+---------------------+------------------------+------------------------+------------------------------+--------------------------+
|2.0410062838810017E10|414047.6090154999       |PRTB                    |3.2497325225799994E9          |2026-06-23 16:37:10.213586|
+---------------------+------------------------+------------------------+------------------------------+--------------------------+



In [23]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/metrica_candidatos_por_cargo"
).show(truncate=False)

+------------+---------------+----------------+--------------------------+
|codigo_cargo|descricao_cargo|total_candidatos|data_referencia           |
+------------+---------------+----------------+--------------------------+
|13          |VEREADOR       |74043           |2026-06-23 16:37:15.526901|
|12          |VICE-PREFEITO  |2169            |2026-06-23 16:37:15.526901|
|11          |PREFEITO       |2137            |2026-06-23 16:37:15.526901|
+------------+---------------+----------------+--------------------------+



In [24]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/metrica_patrimonio_por_partido"
).show(truncate=False)

+--------------+-------------+--------------------+--------------------------+
|numero_partido|sigla_partido|patrimonio_declarado|data_referencia           |
+--------------+-------------+--------------------+--------------------------+
|28            |PRTB         |3.2497325225799994E9|2026-06-23 16:37:21.987459|
|22            |PL           |1.8374646678699996E9|2026-06-23 16:37:21.987459|
|55            |PSD          |1.7996912299499993E9|2026-06-23 16:37:21.987459|
|10            |REPUBLICANOS |1.76640245048E9     |2026-06-23 16:37:21.987459|
|20            |PODE         |1.7286705789999993E9|2026-06-23 16:37:21.987459|
|15            |MDB          |1.5626218354399996E9|2026-06-23 16:37:21.987459|
|44            |UNIÃO        |1.2491080968500004E9|2026-06-23 16:37:21.987459|
|11            |PP           |1.0996781811999998E9|2026-06-23 16:37:21.987459|
|40            |PSB          |8.2207584651E8      |2026-06-23 16:37:21.987459|
|45            |PSDB         |7.831199170500001E8 |2